In [56]:
import spacy
from spacy.matcher import PhraseMatcher
import re
import pandas as pd

In [57]:
pd.set_option('display.max_colwidth', None)
jobs = pd.read_csv("../tech_jobs_clean.csv")
sample_jobs = jobs.sample(300,random_state=42)
sample_jobs_before = sample_jobs.copy(deep=True)

In [58]:
CUE_PATTERN = re.compile(
    r"""
    (?:
        experience\s+(?:with|in) |
        proficiency\s+in |
        knowledge\s+of |
        familiar(?:ity)?\s+with |
        skilled\s+in |
        expertise\s+in |
        working\s+knowledge\s+of |
        hands[-\s]?on\s+experience\s+(?:with|in)
    )
    \s+                                   # whitespace after cue
    (?P<chunk>                             # capture the chunk
        .*?                                # non-greedy
    )
    (?=                                    # stop when we hit a delimiter
        [\.\;\n\r] |                       # period/semicolon/newline
        \u2022 |                           # bullet •
        \s-\s |                            # " - " often used in listings
        $                                   # or end of string
    )
    """,
    re.IGNORECASE | re.VERBOSE | re.DOTALL
)

In [59]:
def extract_skill_chunks_from_description(text: str, max_chunks: int = 10) -> list[str]:
    if pd.isna(text) or not isinstance(text, str) or not text.strip():
        return []

    chunks = []
    for m in CUE_PATTERN.finditer(text):
        chunk = m.group("chunk").strip()
        chunk = re.sub(r"\s+", " ", chunk)
        chunk = chunk.strip(" :,-–—•*")
        if len(chunk) >= 2:
            chunks.append(chunk)
        if len(chunks) >= max_chunks:
            break

    # dedupe preserving order
    seen = set()
    out = []
    for c in chunks:
        k = c.lower()
        if k not in seen:
            seen.add(k)
            out.append(c)
    return out

In [60]:
def add_description_chunks_to_skills_desc(df: pd.DataFrame,
                                         desc_col="description",
                                         skills_col="skills_desc") -> pd.DataFrame:
    # make sure skills_desc exists
    if skills_col not in df.columns:
        df[skills_col] = ""

    def _append(row):
        desc = row.get(desc_col, "")
        existing = row.get(skills_col, "")
        existing = "" if pd.isna(existing) else str(existing)

        chunks = extract_skill_chunks_from_description(desc)
        if not chunks:
            return existing

        chunk_text = "; ".join(chunks)
        return (existing + ("; " if existing.strip() else "") + chunk_text).strip()

    df[skills_col] = df.apply(_append, axis=1)
    return df

sample_jobs = add_description_chunks_to_skills_desc(sample_jobs)

In [61]:
missing_total_before = (
    sample_jobs_before["skills_desc"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Total missing before:", missing_total_before)

Total missing before: 293


In [62]:
sample_jobs_after = sample_jobs_before.copy(deep=True)
sample_jobs_after = add_description_chunks_to_skills_desc(sample_jobs_after)

In [63]:
missing_total_after = (
    sample_jobs_after["skills_desc"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Total missing after:", missing_total_after)

Total missing after: 45


In [64]:
sample_jobs_before["skills_desc"].head(10)

2136     NaN
744      NaN
6842     NaN
12446    NaN
10987    NaN
101      NaN
12766    NaN
10455    NaN
8990     NaN
2584     NaN
Name: skills_desc, dtype: object

In [65]:
sample_jobs_after["skills_desc"].head(10)

2136                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [66]:
before_len = sample_jobs_before["skills_desc"].fillna("").str.len()
after_len = sample_jobs_after["skills_desc"].fillna("").str.len()

print("Average length BEFORE:", before_len.mean())
print("Average length AFTER:", after_len.mean())
print("Average increase:", (after_len - before_len).mean())

Average length BEFORE: 12.01
Average length AFTER: 313.97333333333336
Average increase: 301.9633333333333


In [67]:
sample_jobs_after["extracted_chunks"] = sample_jobs_after["description"].apply(
    extract_skill_chunks_from_description
)

# Flatten all chunks
all_chunks = [chunk for row in sample_jobs_after["extracted_chunks"] for chunk in row]

print("Total chunks extracted:", len(all_chunks))

# Look at most common ones
from collections import Counter
Counter(all_chunks).most_common(15)

Total chunks extracted: 752


[('determining final salary', 2),
 ('engine, transmission, or vehicle facilities or similar industrial manufacturers',
  2),
 ('technical problem-solving methodologies and continuous improvementTime Management – ability to handle and prioritize a large number of simultaneous assignmentsAttention to detail and deadlines, flexibility to adapt to changing work assignmentsOutstanding verbal and written communication skills, interpersonal skills, and ethicsComputer skills using MS Office Suite (Outlook, Word, Excel, Power Point etc',
  2),
 ('grid modernization, renewable energy, energy storage, nuclear power, and fossil fuels',
  2),
 ('the private or public sector', 2),
 ('estimating or scheduling is a plus', 1),
 ('iOS frameworks such as UIKit, Core Data, and Core Animation', 1),
 ('RESTful APIs to connect iOS applications to back-end services', 1),
 ('version control systems such as Git', 1),
 ('SwiftUI', 1),
 ('Firebase or other cloud-based services', 1),
 ('developing and deploying iO

In [68]:
tech_skills = {
    "python", "java", "c++", "c#", "javascript", "typescript",
    "sql", "html", "css",
    "react", "angular", "vue",
    "node.js", "spring", "django", "flask",
    "machine learning", "deep learning", "data analysis",
    "natural language processing", "nlp",
    "numpy", "pandas", "scikit-learn",
    "tensorflow", "keras", "pytorch",
    "linux", "unix", "bash",
    "docker", "kubernetes",
    "aws", "azure", "gcp",
    "git", "ci/cd",
    "autocad", "solidworks",
    "matlab", "simulink",
    "plc", "control systems",
    "six sigma", "lean",
    "quality assurance",
    "project management",
    "agile", "scrum",
    "jira", "confluence",
    "pytest", "junit", "selenium",
    "cypress", "jest", "test automation",
    "unit testing", "integration testing",
    "siemens apogee", "siemens desigo", "c shell", "vsam",
    "angular", "security clearance"
}

In [69]:
def chunk_contains_tech_skill(chunk: str, tech_skills: set[str]) -> bool:
    chunk_lower = chunk.lower()
    for skill in tech_skills:
        if skill in chunk_lower:
            return True
    return False

In [70]:
valid_chunks = [chunk for chunk in all_chunks if chunk_contains_tech_skill(chunk, tech_skills)]
print("Valid chunks that contain tech skills:", len(valid_chunks))
print("Precision:", len(valid_chunks) / len(all_chunks) if all_chunks else 0)

Valid chunks that contain tech skills: 215
Precision: 0.2859042553191489


In [71]:
# display 10 random samples of skills_desc from sample_jobs_after
display(sample_jobs_after[["title", "skills_desc"]].sample(10, random_state=42))

,title,skills_desc
3058,Magnet R&D Engineer,"materials-related industry with an MS/PhD or minimum of 4 years of experience with a BS; (nano-) powder characterization and processing; CAD design in SolidWorks with FEA/Fluid Simulations / FEMM modeling; basic machining techniques, 3D printing and prototype development"
4147,Lead React Developer (15 Years Only),
8510,"Lead Software Engineer, Application Administrator, SIS","source control and source control practices (Git, Subversion, etc"
2584,Test Engineer,"test, software and embedded systems, a passion for working on new hardware, and strong technical skills to deliver future products and prototypes; C/C++ for embedded systemsFamiliarity with data acquisition systems and diagnostic equipment, such as oscilloscopes, bus analyzers, and power suppliesExperience in real-time processing for computer vision and user interaction tasks, high-compute/throughput systems; integration and testing for a complex systemExperience with low level operating systems, RTOS, UBOOT, or other bare-metal programmingExperience coding with peripherals such as UART, SPI, CSI-2, i2c, GPIO, USB, PCIEUnderstanding of DevOps practicesExperience with FPGA and hardware evaluation boards, EDA design tools, and/or ISA simulators"
12131,Senior Software Engineer,"team leadershipBachelor’s Degree in Computer Science or related technical discipline8+ years of experience in Software Engineering; AWSData Pipelines (Spark, AWS EMR)Python & Django experienceReact/NextJS experience; determining final salary"
11952,Developer,"Pega , SQL; integrations like SOAPHas experience in working Agile projectsKnowledge on XML , Java ,SQLGood knowledge of PEGA Guardrails"
16361,Data Engineer,"application and data security concepts, best practices, and common vulnerabilities"
6113,Reliability Engineer - Biotechnology,"technical problem-solving methodologies and continuous improvementTime Management – ability to handle and prioritize a large number of simultaneous assignmentsAttention to detail and deadlines, flexibility to adapt to changing work assignmentsOutstanding verbal and written communication skills, interpersonal skills, and ethicsComputer skills using MS Office Suite (Outlook, Word, Excel, Power Point etc"
101,Seal Product Design Engineer,"engineering drawings and parametric 3D modeling systems; elastomer materials, sealing knowledge or aerospace applications"
11910,"Data Science Reporting Analyst in Columbus, OH",analytics or data science


In [72]:
all_jobs = pd.read_csv('../postings.csv')
all_jobs_before = all_jobs.copy(deep=True)

In [73]:
missing_skills_before = (
    all_jobs_before["skills_desc"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

In [74]:
print("Missing before on full dataset:", missing_skills_before)

Missing before on full dataset: 121410


In [75]:
all_jobs_after = all_jobs_before.copy(deep=True)


In [76]:
all_jobs_after = add_description_chunks_to_skills_desc(all_jobs_after)

In [77]:
missing_skills_after = (
    all_jobs_after["skills_desc"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

In [78]:
print("Missing after on full dataset:", missing_skills_after)

Missing after on full dataset: 39416


Missing Experience Level Values

In [79]:
jobs["formatted_experience_level"].value_counts()

formatted_experience_level
Mid-Senior level    7249
Entry level         3970
Associate           1201
Director             251
Internship           114
Executive             59
Name: count, dtype: int64

In [80]:
EXPERIENCE_LEVEL_MAP = {

    "chief executive officer": "Executive",
    "chief technology officer": "Executive",
    "chief financial officer": "Executive",
    "chief operating officer": "Executive",
    "ceo": "Executive",
    "cto": "Executive",
    "cfo": "Executive",
    "coo": "Executive",
    "vice president": "Executive",
    "vp ": "Executive",
    "general manager": "Executive",

    "director of": "Director",
    "senior director": "Director",
    "director": "Director",
    "head of": "Director",

    "principal": "Mid-Senior level",
    "staff": "Mid-Senior level",
    "lead": "Mid-Senior level",
    "senior": "Mid-Senior level",
    "sr.": "Mid-Senior level",
    "sr ": "Mid-Senior level",
    "manager": "Mid-Senior level",

    "associate": "Associate",
    "assistant": "Associate",

    "internship": "Internship",
    "intern": "Internship",
    "trainee": "Internship",
    "apprentice": "Internship",
}

In [81]:
nlp = spacy.load("en_core_web_sm",
                 disable=["ner","parser"])

In [82]:
def extract_experience_level(title, description, existing_value=None):
    if pd.notna(existing_value) and str(existing_value).strip() != "":
        return existing_value

    title = str(title).lower()
    description = str(description).lower()

    for keyword, level in EXPERIENCE_LEVEL_MAP.items():
        if re.search(r"\b" + re.escape(keyword) + r"\b", title):
            return level

    year_match = re.search(r'(\d+)\+?\s*years?', description)

    if year_match:
        years = int(year_match.group(1))

        if years < 2:
            return "Entry level"
        elif years < 5:
            return "Associate"
        elif years < 10:
            return "Mid-Senior level"
        elif years < 15:
            return "Director"
        else:
            return "Director/Executive"

    return existing_value

In [83]:
sample_jobs['formatted_experience_level'].isna().sum()

np.int64(63)

In [84]:
sample_jobs['formatted_experience_level'].value_counts()

formatted_experience_level
Mid-Senior level    133
Entry level          73
Associate            20
Director              6
Internship            4
Executive             1
Name: count, dtype: int64

In [85]:
sample_jobs["formatted_experience_level"] = sample_jobs.apply(
    lambda row: extract_experience_level(
        str(row["title"]),
        row["formatted_experience_level"]
    ),
    axis=1
)

In [86]:
sample_jobs['formatted_experience_level'].isna().sum()

np.int64(216)

In [87]:
sample_jobs['formatted_experience_level'].value_counts()

formatted_experience_level
Mid-Senior level    72
Director             6
Associate            4
Internship           2
Name: count, dtype: int64

In [88]:
sample_jobs_before.loc[sample_jobs_before['formatted_experience_level'] == 'Mid-Senior level', ['title', 'formatted_experience_level']].sample(10, random_state=42)

,title,formatted_experience_level
4153,Game Developer,Mid-Senior level
5483,Systems Engineer,Mid-Senior level
5304,"Civil/Structural Engineer in Liberty, NC",Mid-Senior level
8686,Thermal Engineer,Mid-Senior level
5425,Sr. SQL Server Database Developer / DBA _NYC / GA,Mid-Senior level
3833,"Product Manager Software Solutions, Advanced",Mid-Senior level
2214,Azure DevOps Engineer,Mid-Senior level
7050,Senior Quality Engineer SDET,Mid-Senior level
10598,Mobile App Testing Engineer,Mid-Senior level
14745,Mainframe Developer,Mid-Senior level


In [89]:
all_jobs_before['formatted_experience_level'].isna().sum()

np.int64(29409)

In [90]:
all_jobs_before['formatted_experience_level'].value_counts()

formatted_experience_level
Mid-Senior level    41489
Entry level         36708
Associate            9826
Director             3746
Internship           1449
Executive            1222
Name: count, dtype: int64

In [91]:
all_jobs_after["formatted_experience_level"] = all_jobs_after.apply(
    lambda row: extract_experience_level(
        row["title"],
        row["description"],
        row["formatted_experience_level"]
    ),
    axis=1
)

In [92]:
all_jobs_after['formatted_experience_level'].isna().sum()

np.int64(10254)

In [93]:
all_jobs_after['formatted_experience_level'].value_counts()

formatted_experience_level
Mid-Senior level      49525
Entry level           37819
Associate             15710
Director               5303
Internship             2067
Director/Executive     1633
Executive              1538
Name: count, dtype: int64

In [94]:
all_jobs_before.loc[all_jobs_before['formatted_experience_level']=='Executive', ['title','formatted_experience_level']].sample(20, random_state=42)

,title,formatted_experience_level
29289,Director of Operations,Executive
50790,Head of Engineering,Executive
6276,Chief People Officer - AI Technology,Executive
100515,Solutions Architect,Executive
60682,"Chief Operating Officer, Property Management",Executive
90933,Registered Nurse 2 CSO W&C's Generalist,Executive
85339,"Vice President, Sales",Executive
42215,General Manager - Printing,Executive
99431,Laboratory Manager Blood Banks Operations FT Days,Executive
91418,Chief Technology Officer,Executive


In [95]:
all_jobs_before.loc[all_jobs_before['formatted_experience_level']=='Associate', ['title','formatted_experience_level']].sample(20, random_state=42)

,title,formatted_experience_level
116778,Marketing Analytics Specialist,Associate
101235,Event Specialist,Associate
11018,Automotive Mobile Glass Technician,Associate
66336,Phlebotomist,Associate
6381,Research Technician III,Associate
30816,Associate District Manager,Associate
119301,Escalations Representative,Associate
54925,"Associate, Strategic Finance",Associate
123082,Data Privacy Analyst,Associate
15163,Assistant General Counsel - Corporate and Securities,Associate


In [96]:
all_jobs_before.loc[all_jobs_before['formatted_experience_level']=='Internship', ['title','formatted_experience_level']].sample(10, random_state=42)

,title,formatted_experience_level
52628,"Intern, ROC Operations",Internship
71326,Intern - Concepts Formulation Engineer,Internship
4517,Auto Glass Installation Technician Trainee,Internship
118972,High School Intern Patient Care Assistant- Flor Medical Surgical 3NW PRN 2nd shift,Internship
9550,Secret Shopper,Internship
60569,Virtual Color Consultation Intern - Summer 2024,Internship
91023,Engineering Intern,Internship
75473,Intern R&D Engineering Operations (On-site),Internship
87840,Apprentice Terminal Operator 12-hr,Internship
111364,Intern - Analytics,Internship


In [97]:
all_jobs_before.loc[all_jobs_before['formatted_experience_level']=='Entry level', ['title','formatted_experience_level']].sample(20, random_state=42)

,title,formatted_experience_level
78169,"Insider Risk Analyst (Remote, MST & PST)",Entry level
71093,Certified Nursing Assistant (CNA),Entry level
96765,"Store Assistant, Full Time",Entry level
120714,Electrical Assembler,Entry level
104849,Electrical Engineer/Electrical PE,Entry level
79024,Maintenance Technician II,Entry level
68237,Recruiter - Contract,Entry level
28025,Plumbing Foreman,Entry level
37305,Crane Service Technician,Entry level
79665,MRI Tech - Hershey Pennsylvania,Entry level


In [98]:
all_jobs_before.loc[all_jobs_before['formatted_experience_level']=='Director', ['title','formatted_experience_level']].sample(20, random_state=42)

,title,formatted_experience_level
113066,Director - Events and Experiential Marketing,Director
40218,"Director, Client Services - Home Based",Director
13968,Director Product Management,Director
123212,Assistant Front Office Manager,Director
114325,"Director, Category Management",Director
71652,"Head of Direct Procurement, Global",Director
4674,"Director Regional Health, Safety, Environmental",Director
14098,Senior Finance Manager - BP Pulse,Director
74638,Sales Director (Wealth),Director
43700,Executive Director Boulder Park Terrace,Director


In [99]:
all_jobs_before.loc[all_jobs_before['formatted_experience_level']=='Mid-Senior level', ['title','formatted_experience_level']].sample(20, random_state=42)

,title,formatted_experience_level
12508,"Registered Nurse (RN)- Outpatient -GI Nurse, Endoscopy Ambulatory Surgery Center, FT",Mid-Senior level
82926,Associate Legal Specialist - Governance and Securities,Mid-Senior level
6431,Development Team Lead,Mid-Senior level
14081,10378 Store Manager,Mid-Senior level
29733,Construction Superintendent,Mid-Senior level
15497,Sr. Web Growth Marketer,Mid-Senior level
112794,Sr. Auditor,Mid-Senior level
104076,Customer Service Representative,Mid-Senior level
42210,Office Administrator,Mid-Senior level
120774,Auditor II (Retail),Mid-Senior level


In [100]:
all_jobs_after.loc[all_jobs_after['formatted_experience_level']=='Internship', ['title','formatted_experience_level']].sample(20, random_state=42)

,title,formatted_experience_level
10346,Police Communications Trainee,Internship
31883,Government Affairs Intern Position – Summer 2024,Internship
79413,Recruiter Trainee,Internship
9073,State & Local Tax Intern - Summer 2025 (San Jose),Internship
1149,Instrumentation Manufacturing Engineer Intern,Internship
8621,Social Media Marketing Communication Intern,Internship
57904,Area Leader Trainee,Internship
34272,Sales Development Representative Intern,Internship
118383,Materials Science Intern,Internship
46434,Summer Internship - Orlowitz Residence,Internship


In [101]:
all_jobs_after.loc[all_jobs_after['formatted_experience_level']=='Entry level', ['title','formatted_experience_level']].sample(10, random_state=42)

,title,formatted_experience_level
115279,Echo Tech,Entry level
62709,FT Produce Sales Associate,Entry level
88712,*Monitor/Telemetry Technician PRN Nights | Pittsburgh LTACH,Entry level
6808,Local Performance Marketing Specialist,Entry level
73213,PennDOT CDL Operator (Transportation Equipment Operator A),Entry level
14435,Dialysis Unit Clerk,Entry level
3633,Customer Sales & Service Rep I – Bilingual Preferred (English/Spanish),Entry level
90743,Guest Service Agent-Kimpton Aertson Hotel,Entry level
15844,Real Estate Development Coordinator,Entry level
95931,Staff Accountant,Entry level


In [102]:
all_jobs_after.loc[all_jobs_after['formatted_experience_level']=='Executive', ['title','formatted_experience_level']].sample(20, random_state=42)

,title,formatted_experience_level
42645,Project Executive,Executive
6276,Chief People Officer - AI Technology,Executive
120017,Chief Nursing Officer (CNO),Executive
57083,Vistage Executive Coach,Executive
112311,Executive Chef,Executive
112666,Distinguished Software Engineer Cryptography and Data Protection (REMOTE),Executive
84348,"Management Information Specialist, CG-0301-14 (ICTAP)",Executive
120611,"Vice President, Business Development",Executive
75245,Market President,Executive
92582,"General Manager - San Jose, CA",Executive


In [103]:
all_jobs_after.loc[all_jobs_after['formatted_experience_level']=='Director', ['title','formatted_experience_level']].sample(20, random_state=42)

,title,formatted_experience_level
12792,Service Technician,Director
80841,Infrastructure Engineer,Director
72382,Criminal Justice School Director,Director
55612,Marketing Director,Director
1532,"Director, Systems Architect (PMIC)",Director
102891,"Associate Director, News and Media Relations",Director
67808,Head of Counterparty Credit Risk Reporting (Hybrid),Director
118386,National Sales Director,Director
31021,Accounting Policy Director,Director
68161,"Director, Prior Authorization Center",Director


In [104]:
all_jobs_after.loc[all_jobs_after['formatted_experience_level']=='Associate', ['title','formatted_experience_level']].sample(20, random_state=42)

,title,formatted_experience_level
52399,Legal Specialist,Associate
90784,Volunteer: Seeking Military Veterans to Serve and Support our Hospice Veterans,Associate
14848,LVN Residency July- Med Surg,Associate
78354,High Level Disinfection I,Associate
51729,Maintenance Engineer,Associate
111180,Resource Efficiency Manager - 1898182,Associate
41592,CONTRACT Epicor Specialist,Associate
100348,1099 Orthopedics Rep,Associate
108685,Underwriting Consultant (Commercial Middle Market),Associate
36218,Part Time - Loader/Cart Associate - Flexible,Associate


In [105]:
all_jobs_after.loc[all_jobs_after['formatted_experience_level']=='Mid-Senior level', ['title','formatted_experience_level']].sample(20, random_state=42)

,title,formatted_experience_level
99788,Quality Configuration Manager,Mid-Senior level
58083,Account Manager - Accelerated Sales Program,Mid-Senior level
64971,Solution Specialist - Security,Mid-Senior level
23758,Data Engineer,Mid-Senior level
52225,Accounting Manager,Mid-Senior level
99801,Senior Sourcing Specialist,Mid-Senior level
23655,Program Manager,Mid-Senior level
75901,Office Manager - Outpatient Addiction Recovery,Mid-Senior level
15162,"Associate Branch Manager (Foward Hire) - Charlotte, NC",Mid-Senior level
53085,Lead Auditor- Banking,Mid-Senior level


In [106]:
import os

os.makedirs('../data/processed', exist_ok=True)

jobs_updated_path = '../data/processed/jobs_updated.csv'
all_jobs_after.to_csv(jobs_updated_path, index=False)
